# 証券営業インテリジェンス ハンズオン
## Part 3: Cortex AI Functions

このノートブックでは、**Snowflake Cortex AI Functions** を使って非構造化データを分析し、証券営業の業務を効率化します。

### このパートで体験できること

| AI Function | 用途 | 証券営業での活用例 |
|---|---|---|
| `AI_COMPLETE` | 自由テキスト生成・要約 | アナリストレポートを3行に要約 / 訪問前ブリーフィング自動作成 |
| `AI_SENTIMENT` | 感情分析 | ニュースがポジティブ/ネガティブか判定 |
| `AI_EXTRACT` | 情報抽出（構造化） | ニュースから銘柄・イベントをJSON抽出 |
| `AI_CLASSIFY` | テキスト分類 | 記事を営業アクション別に分類 |
| `AI_REDACT` | 個人情報マスキング | 顧客メモのPIIを自動消去 |
| `AI_SIMILARITY` | 類似度計算 | 山田様に似た顧客プロファイルを検索 |

### 🚀 アハ体験ポイント

> **「2,000文字のアナリストレポートが3行に。1時間の訪問準備が10秒に。」**
>
> SQL 1行で、営業担当者の日常業務を劇的に効率化できます。

### 前提条件
- `part0_setup.ipynb` 実行済み
- `part1_data_generation.ipynb` 実行済み（100名の顧客データ）


In [ ]:
%%sql -r result_env
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH;

## 対象データの確認

AI Functions を適用する前に、処理対象のデータを確認しましょう。

- **NEWS_ARTICLES**: マーケットニュース 50件（AI_SENTIMENT, AI_EXTRACT, AI_CLASSIFY 対象）
- **ANALYST_REPORTS**: アナリストレポート 30件（AI_COMPLETE で要約 対象）
- **DIM_CUSTOMER**: 顧客マスタ 100名（AI_COMPLETE でブリーフィング生成）


In [ ]:
%%sql -r result_news_summary
-- ニュース記事の件数とカテゴリ分布
SELECT
    CATEGORY AS カテゴリ,
    COUNT(*) AS 件数,
    ROUND(AVG(LENGTH(CONTENT))) AS 平均文字数
FROM NEWS_ARTICLES
GROUP BY CATEGORY
ORDER BY 件数 DESC;

In [ ]:
%%sql -r result_news_sample
-- ニュース記事のサンプル（最初の2件）
SELECT
    NEWS_ID,
    PUBLISH_DATE,
    CATEGORY,
    TITLE,
    LEFT(CONTENT, 200) AS 本文（先頭200字）
FROM NEWS_ARTICLES
ORDER BY PUBLISH_DATE DESC
LIMIT 2;

In [ ]:
%%sql -r result_analyst_sample
-- アナリストレポートのサンプル（トヨタ）
SELECT
    REPORT_ID,
    PUBLISH_DATE,
    SECURITY_NAME,
    ANALYST_NAME,
    RATING AS 投資判断,
    TARGET_PRICE AS 目標株価,
    LEFT(EXECUTIVE_SUMMARY, 200) AS サマリー（先頭200字）
FROM ANALYST_REPORTS
WHERE SECURITY_CODE = '7203'
ORDER BY PUBLISH_DATE DESC
LIMIT 2;

## Step 1: AI_COMPLETE - テキスト要約・自動生成

### AI_COMPLETE とは
- カスタムプロンプトで自由にテキストを生成する関数
- プロンプトエンジニアリングにより出力を細かく制御可能
- 使い方: `SNOWFLAKE.CORTEX.COMPLETE(model, prompt)`

> ⏱️ **目安: 10分**


### 1-1. アナリストレポートの瞬時要約

アナリストレポートは平均 **2,000〜3,000文字** の長文です。
AI_COMPLETE に要約プロンプトを渡すだけで、**3行の要約** に自動変換できます。

> **💡 アハ体験**: 読むのに5分かかるレポートが3秒で要約される！


In [ ]:
%%sql -r result_summarize
-- AI_COMPLETE: アナリストレポートを3行に要約
SELECT
    SECURITY_NAME AS 銘柄,
    ANALYST_NAME AS アナリスト,
    RATING AS 投資判断,
    TARGET_PRICE AS 目標株価,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-6',
        '以下のアナリストレポートを3行で要約してください。投資判断に重要な情報を中心に簡潔にまとめること。\n' ||
        EXECUTIVE_SUMMARY || ' ' || INVESTMENT_THESIS || ' ' || KEY_RISKS
    ) AS AI要約
FROM ANALYST_REPORTS
WHERE SECURITY_CODE = '7203'
ORDER BY PUBLISH_DATE DESC
LIMIT 3;

### 1-2. AI_COMPLETE の活用パターン

| | シンプル（要約） | 高度なカスタマイズ |
|---|---|---|
| プロンプト例 | `'3行で要約: ' \|\| text` | 書式指定・複数情報統合・多言語対応 |
| 出力の自由度 | 基本的な要約 | プロンプト次第で何でも |
| 使いやすさ | 比較的シンプル | 高度なカスタマイズが可能 |

**ポイント**: `AI_COMPLETE` でプロンプトを工夫することで、要約からカスタム書式レポート生成まで柔軟に対応可能。


In [ ]:
%%sql -r result_complete_format
-- AI_COMPLETE: 特定の書式でレポートを要約（出力フォーマットを指定）
SELECT
    SECURITY_NAME AS 銘柄,
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            '以下のアナリストレポートを、証券営業担当者向けに以下の形式で要約してください。
',
            '【投資判断】: （買い/中立/売りと根拠を1行で）
',
            '【ポイント】: （最重要ポイントを3つ、箇条書き）
',
            '【リスク】: （最大のリスクを1行で）

',
            'レポート:
', EXECUTIVE_SUMMARY, '

投資テーマ:
', INVESTMENT_THESIS,
            '

リスク要因:
', KEY_RISKS
        )
    ) AS フォーマット済み要約
FROM ANALYST_REPORTS
WHERE SECURITY_CODE = '7203'
ORDER BY PUBLISH_DATE DESC
LIMIT 1;

### 1-3. 訪問前ブリーフィング自動生成（アハ体験！）

顧客IDを指定するだけで、**訪問前に必要な全情報をまとめたブリーフィングシート** を自動生成します。

> **💡 アハ体験**: 以前は複数システムを見て1時間かけて準備していた訪問ブリーフィングが、**SQL 1本で10秒以内に**生成できます！


In [ ]:
%%sql -r result_briefing
-- 顧客C001（山田太郎様）の訪問前ブリーフィング自動生成
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'mistral-large2',
    CONCAT(
        '以下の顧客情報をもとに、証券営業担当者向けの訪問前ブリーフィングを作成してください。
',
        '## 訪問前ブリーフィング
',
        '【顧客基本情報】
',
        '氏名: ', c.CUSTOMER_NAME, '
',
        '年齢: ', c.AGE, '歳 / 職業: ', c.OCCUPATION, '
',
        '総資産: ', TO_CHAR(c.TOTAL_ASSETS, '999,999,999,999'), '円 / ',
        'セグメント: ', c.SEGMENT, '
',
        '担当RM: ', c.RM_NAME, ' / 最終接触: ', c.LAST_CONTACT_DATE, '

',
        '【ライフイベント】
',
        LISTAGG(e.EVENT_TYPE || ': ' || e.DESCRIPTION || '（緊急度: ' || e.URGENCY || '）', '
')
            WITHIN GROUP (ORDER BY e.EVENT_DATE DESC), '

',
        '【注意事項】
',
        '・今回の相談: 孫の海外留学費用2,000万円が必要
',
        '・トヨタ株（7203）の売却を検討中

',
        '上記をもとに、本日の訪問で確認すべき事項と提案方向性を日本語でまとめてください。'
    )
) AS 訪問前ブリーフィング
FROM DIM_CUSTOMER c
LEFT JOIN DIM_LIFE_EVENT e ON c.CUSTOMER_ID = e.CUSTOMER_ID
WHERE c.CUSTOMER_ID = 'C001'
GROUP BY c.CUSTOMER_NAME, c.AGE, c.OCCUPATION, c.TOTAL_ASSETS, c.SEGMENT, c.RM_NAME, c.LAST_CONTACT_DATE;

## Step 2: AI_SENTIMENT - 感情分析

`AI_SENTIMENT` は、テキストの感情（ポジティブ/ネガティブ/ニュートラル）を自動判定します。

**証券営業での活用例:**
- ニュース記事が保有銘柄にポジティブかネガティブかを自動判定
- 顧客からのクレームメールを優先度付きで処理
- 市場全体のセンチメントをリアルタイムで把握

> ⏱️ **目安: 10分**


In [ ]:
%%sql -r result_sentiment
-- ニュース記事の感情分析（上位10件）
SELECT
    NEWS_ID,
    PUBLISH_DATE AS 発行日,
    LEFT(TITLE, 60) AS タイトル,
    RELATED_SECURITIES AS 関連銘柄,
    SNOWFLAKE.CORTEX.AI_SENTIMENT(CONTENT):label::VARCHAR AS AI感情分析
FROM NEWS_ARTICLES
ORDER BY PUBLISH_DATE DESC
LIMIT 10;

In [ ]:
%%sql -r result_sentiment_summary
-- 感情分析サマリー: カテゴリ × センチメント別の集計
SELECT
    CATEGORY AS カテゴリ,
    SNOWFLAKE.CORTEX.AI_SENTIMENT(CONTENT):label::VARCHAR AS センチメント,
    COUNT(*) AS 件数
FROM NEWS_ARTICLES
GROUP BY CATEGORY, センチメント
ORDER BY CATEGORY, 件数 DESC;

### 2-2. 顧客ポートフォリオ連携ニュースアラート

山田様（C001）が保有する銘柄に関するニュースを自動収集し、感情分析を実施します。

> **💡 実践的な活用**: 毎朝、担当顧客の保有銘柄に関するネガティブニュースをアラートとして受け取る仕組みが作れます。


In [ ]:
%%sql -r result_portfolio_alert
-- C001山田様の保有銘柄のニュース感情アラート
WITH yamada_holdings AS (
    SELECT DISTINCT SECURITY_CODE, SECURITY_NAME, MARKET_VALUE
    FROM FACT_PORTFOLIO
    WHERE CUSTOMER_ID = 'C001'
),
news_sentiment AS (
    SELECT
        n.NEWS_ID,
        n.PUBLISH_DATE,
        n.TITLE,
        n.RELATED_SECURITIES,
        n.CONTENT,
        SNOWFLAKE.CORTEX.AI_SENTIMENT(n.CONTENT):label::VARCHAR AS センチメント
    FROM NEWS_ARTICLES n
)
SELECT
    h.SECURITY_NAME AS 保有銘柄,
    TO_CHAR(h.MARKET_VALUE, '999,999,999') AS 時価評価額,
    ns.PUBLISH_DATE AS ニュース日付,
    LEFT(ns.TITLE, 60) AS ニュースタイトル,
    ns.センチメント,
    CASE ns.センチメント
        WHEN 'negative' THEN '⚠️ 要確認'
        WHEN 'positive' THEN '✅ 好材料'
        ELSE '📋 中立'
    END AS アクション
FROM yamada_holdings h
JOIN news_sentiment ns ON ns.RELATED_SECURITIES LIKE '%' || h.SECURITY_CODE || '%'
ORDER BY h.MARKET_VALUE DESC, ns.PUBLISH_DATE DESC;

## Step 3: AI_EXTRACT - 構造化情報の抽出

`AI_EXTRACT` は、非構造化テキストから指定した項目を **JSON形式** で抽出します。

**証券営業での活用例:**
- ニュース記事から関連銘柄・イベントタイプ・影響方向を自動抽出
- 決算短信から売上・営業利益・配当情報を構造化
- 顧客との会話メモから重要情報を抽出

> ⏱️ **目安: 10分**


In [ ]:
%%sql -r result_extract
-- AI_EXTRACT: ニュース記事から銘柄・イベント情報を構造化抽出
SELECT
    NEWS_ID,
    LEFT(TITLE, 50) AS タイトル,
    SNOWFLAKE.CORTEX.AI_EXTRACT(
        CONTENT,
        OBJECT_CONSTRUCT(
            'related_stocks', '記事に関連する企業名・銘柄コードのリスト（例: [トヨタ自動車 7203, ソニー 6758]）',
            'event_type', '記事のイベント種別（決算発表/経営戦略/マクロ経済/政策/その他）',
            'impact_direction', '株価への影響方向（ポジティブ/ネガティブ/ニュートラル/不明）',
            'urgency', '証券営業担当者が顧客へ連絡すべき緊急度（高/中/低）',
            'action_required', '担当者が取るべき具体的なアクション（1文以内）'
        )
    ) AS 抽出結果
FROM NEWS_ARTICLES
ORDER BY PUBLISH_DATE DESC
LIMIT 5;

### 3-2. アナリストレポートのリスク要因を構造化

アナリストレポートのリスク要因を構造化して、顧客説明に使いやすい形式に整えます。


In [ ]:
%%sql -r result_extract_risk
-- AI_EXTRACT: アナリストレポートからリスク要因を抽出
SELECT
    SECURITY_NAME AS 銘柄,
    RATING AS 投資判断,
    SNOWFLAKE.CORTEX.AI_EXTRACT(
        KEY_RISKS,
        OBJECT_CONSTRUCT(
            'risk_level', 'リスクの全体評価（高/中/低）',
            'main_risks', '主要リスク要因のリスト（最大3つ）',
            'timeframe', 'リスクが顕在化する時間軸（短期3ヶ月以内/中期6ヶ月/長期1年以上）',
            'hedge_suggestion', 'リスクヘッジのための提案（1文）'
        )
    ) AS リスク分析
FROM ANALYST_REPORTS
WHERE SECURITY_CODE = '7203'
ORDER BY PUBLISH_DATE DESC
LIMIT 2;

## Step 4: AI_CLASSIFY - テキスト分類

`AI_CLASSIFY` は、テキストを指定したカテゴリに分類します。

**証券営業での活用例:**
- ニュースを「どの営業アクションに使えるか」でタグ付け
- 顧客クレームを種別分類して担当部署に振り分け
- 商品提案の優先度分類

> ⏱️ **目安: 10分**


In [ ]:
%%sql -r result_classify
-- AI_CLASSIFY: ニュースを営業活動タイプ別に分類
SELECT
    NEWS_ID,
    PUBLISH_DATE AS 発行日,
    LEFT(TITLE, 55) AS タイトル,
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        CONTENT,
        ARRAY_CONSTRUCT(
            '株式売買タイミング提案',
            '相続・贈与対策提案',
            '証券担保ローン提案',
            'ポートフォリオ見直し提案',
            '新商品・ファンド提案',
            '情報提供のみ（アクション不要）'
        )
    ):label::VARCHAR AS 営業アクション分類
FROM NEWS_ARTICLES
ORDER BY PUBLISH_DATE DESC
LIMIT 10;

In [ ]:
%%sql -r result_classify_agg
-- 分類結果の集計: どのアクションが多いか
SELECT
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        CONTENT,
        ARRAY_CONSTRUCT(
            '株式売買タイミング提案',
            '相続・贈与対策提案',
            '証券担保ローン提案',
            'ポートフォリオ見直し提案',
            '新商品・ファンド提案',
            '情報提供のみ（アクション不要）'
        )
    ):label::VARCHAR AS 営業アクション,
    COUNT(*) AS 件数
FROM NEWS_ARTICLES
GROUP BY 営業アクション
ORDER BY 件数 DESC;

## Step 5: AI_REDACT - 個人情報の自動マスキング

`AI_REDACT` は、テキスト中の**個人情報（PII）を自動的に検出し、マスキング**します。

**金融機関での活用理由:**
- コンプライアンス対応: 個人情報保護法 / GDPR
- AIモデルへの入力時に個人情報を除去
- 社内ログからの個人情報漏洩防止

**検出対象**: 氏名・電話番号・住所・メールアドレス・マイナンバー・クレジットカード番号など

> ⏱️ **目安: 5分**


In [ ]:
%%sql -r result_redact
-- AI_REDACT: 顧客問い合わせメモから個人情報を自動マスキング
-- ※実際の業務では、顧客メモをAIに入力する前にマスキングすることで情報漏洩を防止

WITH sample_notes AS (
    SELECT 'C001' AS CUSTOMER_ID,
           '山田太郎様（090-1234-5678）から着信あり。住所：東京都港区赤坂1-2-3 山田マンション501号。マイナンバー1234-5678-9012にて本人確認済み。次回訪問は2025年4月10日午後2時にご自宅にて。' AS CONTACT_NOTE
    UNION ALL
    SELECT 'C002',
           '佐藤花子様 Tel:03-5678-9012 よりご連絡。神奈川県横浜市中区本町4-5-6。生年月日：1965年3月15日。本日15:00にご来店予定。クレカ番号4111-1111-1111-1111確認済み。'
    UNION ALL
    SELECT 'C003',
           '鈴木一郎様からメール受信。アドレス: suzuki.ichiro@example.com。ポートフォリオの見直しについてご相談希望。総資産3億円のうち現金1.5億円を運用したいとのこと。'
)
SELECT
    CUSTOMER_ID,
    CONTACT_NOTE AS 元のメモ,
    SNOWFLAKE.CORTEX.AI_REDACT(CONTACT_NOTE)::VARCHAR AS マスキング済みメモ
FROM sample_notes;

## Step 6: AI_SIMILARITY - 類似顧客の検索

`AI_SIMILARITY` は、2つのテキストの**意味的な類似度**（0〜1）を返します。

**証券営業での活用例:**
- 山田様と似たプロファイルの顧客を発見 → 同じ提案が有効
- 過去に成功した提案事例に近い顧客を発見
- 顧客セグメンテーションの自動化

> ⏱️ **目安: 5分**


In [ ]:
%%sql -r result_similarity
-- AI_SIMILARITY: 山田様に似たプロファイルの顧客を検索
-- 山田様のプロファイル: 68歳・元上場企業役員・8億円・低リスク・相続対策
WITH yamada AS (
    SELECT
        RISK_TOLERANCE || ' / ' || INVESTMENT_PURPOSE ||
        ' / 年齢:' || AGE || '歳 / 資産:' || TO_CHAR(TOTAL_ASSETS/100000000, '9.0') || '億円' AS PROFILE
    FROM DIM_CUSTOMER WHERE CUSTOMER_ID = 'C001'
)
SELECT
    c.CUSTOMER_ID,
    c.CUSTOMER_NAME,
    c.AGE AS 年齢,
    c.RISK_TOLERANCE AS リスク許容度,
    c.INVESTMENT_PURPOSE AS 投資目的,
    TO_CHAR(c.TOTAL_ASSETS/100000000, '9.0') AS 資産億円,
    ROUND(SNOWFLAKE.CORTEX.AI_SIMILARITY(
        (SELECT PROFILE FROM yamada),
        c.RISK_TOLERANCE || ' / ' || c.INVESTMENT_PURPOSE ||
        ' / 年齢:' || c.AGE || '歳 / 資産:' || TO_CHAR(c.TOTAL_ASSETS/100000000, '9.0') || '億円'
    ), 3) AS 類似度スコア
FROM DIM_CUSTOMER c
WHERE c.CUSTOMER_ID != 'C001'
ORDER BY 類似度スコア DESC
LIMIT 10;

## Step 7: 総合活用 - 山田様への提案材料を自動生成

これまでのAI Functionsをすべて組み合わせて、**山田様（C001）への提案に関連するニュースを一括処理**します。

> **💡 アハ体験**: 各種AI Functionsを連鎖させることで、「ニュース取得 → 感情分析 → 情報抽出 → 分類 → 営業アクション決定」というパイプラインが **SQL 1本** で完成します！

> ⏱️ **目安: 5分**


In [ ]:
%%sql -r result_integrated
-- 山田様（C001）関連ニュースの一括AI処理パイプライン
WITH yamada_securities AS (
    SELECT DISTINCT SECURITY_CODE
    FROM FACT_PORTFOLIO
    WHERE CUSTOMER_ID = 'C001'
),
relevant_news AS (
    SELECT n.*
    FROM NEWS_ARTICLES n, yamada_securities ys
    WHERE n.RELATED_SECURITIES LIKE '%' || ys.SECURITY_CODE || '%'
       OR n.CATEGORY IN ('マクロ経済', '税制・制度', '金融市場')
)
SELECT
    rn.PUBLISH_DATE AS 日付,
    LEFT(rn.TITLE, 50) AS タイトル,
    -- 感情分析
    SNOWFLAKE.CORTEX.AI_SENTIMENT(rn.CONTENT):label::VARCHAR AS センチメント,
    -- 営業アクション分類
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        rn.CONTENT,
        ARRAY_CONSTRUCT(
            '株式売買タイミング提案', '相続・贈与対策提案', '証券担保ローン提案',
            'ポートフォリオ見直し提案', '情報提供のみ'
        )
    ):label::VARCHAR AS 営業アクション,
    -- 緊急度抽出
    SNOWFLAKE.CORTEX.AI_EXTRACT(rn.CONTENT,
        OBJECT_CONSTRUCT('urgency', '顧客への連絡緊急度（高/中/低）')
    ):urgency::VARCHAR AS 緊急度
FROM relevant_news rn
ORDER BY rn.PUBLISH_DATE DESC;

## やってみよう！ - 応用練習

以下のSQLを参考に、自分でAI Functionsを試してみましょう。

### 練習 1: 別の銘柄のアナリストレポートを要約してみよう
証券コード `7011`（三菱重工業）または `6758`（ソニーグループ）のレポートを要約してみてください。


In [ ]:
%%sql -r result_practice1
-- 練習1: TODO - 銘柄コードを変えて試してみよう
-- ヒント: WHERE SECURITY_CODE = '7203' の部分を変更する
SELECT
    SECURITY_NAME AS 銘柄,
    RATING AS 投資判断,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-6',
        '以下のアナリストレポートを3行で要約してください: ' ||
        EXECUTIVE_SUMMARY || ' ' || INVESTMENT_THESIS
    ) AS AI要約
FROM ANALYST_REPORTS
WHERE SECURITY_CODE = '7203'  -- ← ここを変えてみよう！
ORDER BY PUBLISH_DATE DESC
LIMIT 1;

### 練習 2: 自分の担当顧客のブリーフィングを生成しよう

顧客IDを `C001` から別のIDに変えて、異なる顧客のブリーフィングを生成してみましょう。

```sql
-- 顧客IDを変えるだけで、別の顧客のブリーフィングが生成される
WHERE c.CUSTOMER_ID = 'C001'  -- ← C002, C003 など別のIDに変えてみよう
```

### 練習 3: カスタム分類カテゴリを作ってみよう

AI_CLASSIFY の分類カテゴリは自由に設定できます。
自社のビジネスに合ったカテゴリを追加してみましょう。


## まとめ

Part 3 完了！Cortex AI Functions を体験しました。

### 使用した AI Functions

| 関数 | 実行内容 | 証券営業での価値 |
|---|---|---|
| `AI_COMPLETE` | アナリストレポートを3行に要約 / 訪問前ブリーフィング自動生成 | 情報収集・訪問準備時間を大幅削減 |
| `AI_SENTIMENT` | ニュース50件を一括感情分析 | ポートフォリオへの影響を即把握 |
| `AI_EXTRACT` | ニュースから銘柄・イベントをJSON抽出 | 非構造化→構造化データへ |
| `AI_CLASSIFY` | ニュースを営業アクション別に分類 | 優先対応が必要な記事を即特定 |
| `AI_REDACT` | 顧客メモから個人情報をマスキング | コンプライアンス対応を自動化 |
| `AI_SIMILARITY` | 類似プロファイルの顧客を発見 | 成功事例の横展開 |

### 重要なポイント

1. **SQL 1行** で高度なAI処理が完結（外部APIやPythonコード不要）
2. **100行のデータを一括処理** → 従来は人手で不可能なスケール
3. **データと同じ場所でAI処理** → データ移動なし、セキュリティ確保

**次のステップ:** `part5_cortex_analyst.ipynb` で Cortex Analyst（自然言語 to SQL）を体験してください。
